In [1]:
import jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", False)

In [ ]:

import os, json, numpy as np, matplotlib.pyplot as plt
import sys

sys.path.insert(0, '../src')
sys.path.insert(0, '../scripts/experiments_large_scale/')

import cifar10_LR
import importlib
importlib.reload(cifar10_LR)

from cifar10_LR import (
            seed_everything, get_device, build_model, get_transforms, load_cifar10,
            extract_embeddings, save_embeddings, load_embeddings,
            stratified_equal_halves, evaluate_plan
)


In [ ]:

cfg = {
            "root": "/scratch/gpfs/DATASETS/cifar",
            "weights_path": "/home/ph3641/OptimalLowRank/Monge_Conjugation/models/resnet18-f37072fd.pth",
            "device": "cuda",     # "cpu" or "cuda"
            "batch_size": 256,
            "num_workers": 4,
            "model": "resnet18",  # "resnet18", "resnet50", "inception_v3"
            "seed": 123,
            "reg": 0.05,          # Sinkhorn entropic regularization
            "rank": 10,           # rank
            "reuse_embeddings": "",  # path to .npz if you want to reuse
            "save_embeddings": True,
            "run_methods": ["lr_monge"],  # choose which to run
        }

seed_everything(cfg["seed"])
device = get_device(cfg["device"])

if cfg["reuse_embeddings"] and os.path.isfile(cfg["reuse_embeddings"]):
    feats, labels, meta = load_embeddings(cfg["reuse_embeddings"])
    print(f"Loaded embeddings: {feats.shape}, labels: {labels.shape}")
else:
    model, embed_dim, transform = build_model(cfg["model"], device, weights_path=cfg["weights_path"])
    transform = get_transforms(cfg["model"])
    ds = load_cifar10(cfg["root"], transform, download=True)
    print("Dataset size:", len(ds))
    feats, labels = extract_embeddings(ds, model, device, cfg["batch_size"], cfg["num_workers"])
    meta = {"model": cfg["model"], "embed_dim": int(feats.shape[1])}
    print("Embeddings:", feats.shape, "Labels:", labels.shape)
    if cfg["save_embeddings"]:
        out_npz = f"embeddings_{cfg['model']}.npz"
        save_embeddings(out_npz, feats, labels, meta)
        print("Saved", out_npz)


In [ ]:

importlib.reload(cifar10_LR)

A_idx, B_idx = stratified_equal_halves(labels, seed=cfg["seed"])
XA, yA = feats[A_idx], labels[A_idx]
YB, yB = feats[B_idx], labels[B_idx]
print(f"Split sizes: A={XA.shape[0]}, B={YB.shape[0]}")

# Optional: PCA to 50 dims (similar to Zhuang & Chen '24)
from sklearn.decomposition import PCA
pca = PCA(n_components=50, random_state=cfg["seed"])
XA = pca.fit_transform(XA)
YB = pca.transform(YB)
print("Finished PCA")


In [ ]:
import cifar10_eval
import distance_utils
import monge_rotate_lr
sys.path.insert(0, '../src/HiRef/')
import HiRef

with jax.default_device(jax.devices("gpu")[0]):
    XA = jnp.asarray(XA)
    YB = jnp.asarray(YB)

In [ ]:
from cifar10_eval import evaluate_factors
import cifar10_LR
import FRLC.FRLC as frlc
importlib.reload(cifar10_LR)
importlib.reload(frlc)
importlib.reload(monge_rotate_lr)

methods = [ "frlc","lot", "monge_conj"]

#k=1000
#_XA, _YB, _yA, _yB = XA[:k], YB[:k], yA[:k], yB[:k]

results = cifar10_LR.run_all_methods(
    XA, YB, yA, yB,
    methods=methods,
    rank=cfg["rank"],
    reg=cfg["reg"],
    ot_solver="HiRef",
    device=str(device),
    evaluate_factors_fn=evaluate_factors
)

import json
try:
    import jax.numpy as jnp
    JAX_ARRAY_TYPES = (jnp.ndarray,)
except Exception:
    JAX_ARRAY_TYPES = tuple()

def np_encoder(obj):
    # numpy scalars -> Python scalars
    if isinstance(obj, np.generic):
        return obj.item()
    # numpy arrays or jax arrays -> lists
    if isinstance(obj, (np.ndarray,) + JAX_ARRAY_TYPES):
        return obj.tolist()
    # anything else: let json handle or raise
    raise TypeError(f"Object of type {obj.__class__.__name__} is not JSON serializable")

# usage in your loop:
for k, v in results.items():
    if "result" in v:
        print(k, json.dumps(v["result"], indent=2, default=np_encoder))
        if "metrics" in v:
            # optionally drop the big matrix to keep logs short
            metrics_for_log = {kk: vv for kk, vv in v["metrics"].items()
                               if kk != "class_mass_matrix"}
            print(k, "metrics:", json.dumps(metrics_for_log, indent=2, default=np_encoder))
    else:
        print(k, "->", v)
